# Bài 3: Naive Bayes - sáu bài tập

**Họ tên:** Huỳnh Hữu Phúc  
**MSSV:** 2474802010314  
Mỗi bài bên dưới cũng có tệp `.py` độc lập trong `baitap03/`.

## Bài tập 1. Bayes với ba nguyên nhân

Dùng xác suất toàn phần rồi chuẩn hóa ba nhánh; tổng hậu nghiệm phải bằng 1.

In [1]:
"""Bài 1: Bayes với ba nguyên nhân (mở rộng d1_bayes_tay.py)."""

ten_nhom = ["Hoc deu", "Hoc don", "It hoc"]
tien_nghiem = [0.50, 0.30, 0.20]
kha_nang_qua = [0.95, 0.70, 0.30]

# Xác suất qua môn chung là tổng của ba nhánh.
p_qua = sum(p * q for p, q in zip(tien_nghiem, kha_nang_qua))
hau_nghiem = [p * q / p_qua for p, q in zip(tien_nghiem, kha_nang_qua)]

print(f"P(qua mon) = {p_qua:.4f}")
for ten, p in zip(ten_nhom, hau_nghiem):
    print(f"P({ten} | qua mon) = {p:.4f}")
print(f"Tong ba xac suat = {sum(hau_nghiem):.4f}")


P(qua mon) = 0.7450
P(Hoc deu | qua mon) = 0.6376
P(Hoc don | qua mon) = 0.2819
P(It hoc | qua mon) = 0.0805
Tong ba xac suat = 1.0000


**Nhận xét:** P(qua môn)=0,7450. Hậu nghiệm của ba nhóm học đều, học dồn, ít học là 0,6376; 0,2819; 0,0805 và cộng lại bằng 1.

## Bài tập 2. Từ nào tố cáo mạnh nhất

Hiệu `feature_log_prob_[1] - feature_log_prob_[0]` cho biết từ nghiêng về lớp nào.

In [2]:
"""Bài 2: lấy hiệu feature_log_prob_ để tìm từ đặc trưng cho hai lớp."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
mo_hinh = MultinomialNB(alpha=1.0).fit(X, df["la_rac"])
do_chenh = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]
bang = pd.DataFrame({"tu": vec.get_feature_names_out(), "log(P_rac/P_thuong)": do_chenh})

print("5 tu cho thay tin rac manh nhat (chenh log duong):")
print(bang.sort_values(["log(P_rac/P_thuong)", "tu"], ascending=[False, True]).head(5).to_string(index=False))
print()
print("5 tu cho thay tin thuong manh nhat (chenh log am):")
print(bang.sort_values(["log(P_rac/P_thuong)", "tu"], ascending=[True, True]).head(5).to_string(index=False))
print("Nhan xet: do chenh duong nghieng ve rac; am nghieng ve thuong.")


5 tu cho thay tin rac manh nhat (chenh log duong):
   tu  log(P_rac/P_thuong)
  mãi             3.203435
  tay             2.905600
 ngay             2.661837
 miễn             2.479516
trúng             2.374155

5 tu cho thay tin thuong manh nhat (chenh log am):
   tu  log(P_rac/P_thuong)
  lớp            -2.612187
   em            -2.561756
  bài            -2.294553
  báo            -2.274074
phòng            -2.274074
Nhan xet: do chenh duong nghieng ve rac; am nghieng ve thuong.


**Nhận xét:** Năm từ nghiêng về rác nhất: mãi, tay, ngay, miễn, trúng. Năm từ nghiêng về thường nhất: lớp, em, bài, báo, phòng.

## Bài tập 3. Đổi tham số làm trơn

So sánh accuracy, precision, recall với năm giá trị alpha.

In [3]:
"""Bài 3: đổi alpha của bộ lọc và so sánh ba thước đo."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
tin_train, tin_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"],
)
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(tin_train)
X_test = vec.transform(tin_test)

print(f"{'alpha':>8s} {'accuracy':>10s} {'precision':>10s} {'recall':>8s}")
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    mo_hinh = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    y_pred = mo_hinh.predict(X_test)
    print(f"{alpha:8.3f} {accuracy_score(y_test, y_pred):10.4f} "
          f"{precision_score(y_test, y_pred, zero_division=0):10.4f} "
          f"{recall_score(y_test, y_pred):8.4f}")
print("Nhan xet: trong 5 muc alpha tren, ba chi so tren tap test khong doi.")
print("Ve nguyen tac, alpha rat lon keo xac suat tu cua hai lop lai gan nhau;")
print("khi do bang chung tu yeu va prior lop thuong chi phoi nhieu hon.")


   alpha   accuracy  precision   recall
   0.001     0.9467     1.0000   0.8667
   0.100     0.9467     1.0000   0.8667
   1.000     0.9467     1.0000   0.8667
  10.000     0.9467     1.0000   0.8667
 100.000     0.9467     1.0000   0.8667
Nhan xet: trong 5 muc alpha tren, ba chi so tren tap test khong doi.
Ve nguyen tac, alpha rat lon keo xac suat tu cua hai lop lai gan nhau;
khi do bang chung tu yeu va prior lop thuong chi phoi nhieu hon.


**Nhận xét:** Trong năm giá trị thử, cả ba thước đo trên tập kiểm tra giữ nguyên. Khi alpha rất lớn, xác suất từ của hai lớp tiến gần nhau nên vai trò của từ yếu đi và prior chi phối hơn.

## Bài tập 4. Tự viết bộ lọc

Tự đếm từ và cộng log xác suất; đối chiếu trực tiếp với `MultinomialNB`.

In [4]:
"""Bài 4: tự đếm từ và tính log điểm Naive Bayes trên đủ 300 tin."""

from collections import Counter
from math import log

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
tin_train, tin_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"],
)

# Cùng quy tắc r"(?u)\S+" của CountVectorizer: chuyển chữ thường, tách khoảng trắng.
def tach_tu(cau):
    return cau.lower().split()


tu_dien = {tu for cau in tin_train for tu in tach_tu(cau)}
V = len(tu_dien)
dem_lop = Counter(y_train)
dem_tu = {0: Counter(), 1: Counter()}
for cau, nhan in zip(tin_train, y_train):
    dem_tu[int(nhan)].update(tach_tu(cau))
tong_tu = {nhan: sum(dem_tu[nhan].values()) for nhan in [0, 1]}


def diem_log(cau, nhan, alpha=1.0):
    """log P(nhãn) + tổng số lần từ xuất hiện nhân log P(từ|nhãn)."""
    diem = log(dem_lop[nhan] / len(y_train))
    for tu, so_lan in Counter(tach_tu(cau)).items():
        if tu in tu_dien:  # Từ mới bị bỏ qua, giống vec.transform.
            xac_suat = (dem_tu[nhan][tu] + alpha) / (tong_tu[nhan] + alpha * V)
            diem += so_lan * log(xac_suat)
    return diem


du_doan_tu_viet = [int(diem_log(cau, 1) > diem_log(cau, 0)) for cau in tin_test]
acc_tu_viet = accuracy_score(y_test, du_doan_tu_viet)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(tin_train)
X_test = vec.transform(tin_test)
mo_hinh = MultinomialNB(alpha=1.0).fit(X_train, y_train)
du_doan_thu_vien = mo_hinh.predict(X_test)
acc_thu_vien = accuracy_score(y_test, du_doan_thu_vien)
so_du_doan_khac = sum(a != b for a, b in zip(du_doan_tu_viet, du_doan_thu_vien))

print(f"So mau train/test: {len(tin_train)}/{len(tin_test)}")
print(f"So tu trong tu dien tu viet: {V}")
print(f"So tu trong tu dien thu vien: {len(vec.vocabulary_)}")
print(f"Accuracy tu viet: {acc_tu_viet:.4f}")
print(f"Accuracy MultinomialNB: {acc_thu_vien:.4f}")
print(f"So du doan khac nhau: {so_du_doan_khac}")
print("Giai thich: hai cach dung cung train/test, quy tac tach tu, prior, alpha=1")
print("va bo qua tu ngoai tu dien. Cong log tranh tran duoi khi tin dai.")


So mau train/test: 225/75
So tu trong tu dien tu viet: 40
So tu trong tu dien thu vien: 40
Accuracy tu viet: 0.9467
Accuracy MultinomialNB: 0.9467
So du doan khac nhau: 0
Giai thich: hai cach dung cung train/test, quy tac tach tu, prior, alpha=1
va bo qua tu ngoai tu dien. Cong log tranh tran duoi khi tin dai.


**Nhận xét:** Cả hai cách đạt accuracy=0,9467 và có 0 dự đoán khác nhau. Bộ tự viết dùng prior của train, 40 từ học từ train, alpha=1 và cộng log để tránh tràn dưới. Nếu kết quả lệch, cần kiểm tra quy tắc tách từ, dữ liệu dùng học từ điển, prior và cách xử lý từ mới.

## Bài tập 5. Chọn sai biến thể Naive Bayes

Chạy MultinomialNB trên dữ liệu liên tục và giải thích vì sao giả định không phù hợp.

In [5]:
"""Bài 5: cố ý dùng sai MultinomialNB cho hai đặc trưng liên tục."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB


df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y,
)
for ten, mo_hinh in [
    ("MultinomialNB", MultinomialNB()),
    ("GaussianNB", GaussianNB()),
]:
    mo_hinh.fit(X_train, y_train)
    print(f"{ten}: accuracy = {accuracy_score(y_test, mo_hinh.predict(X_test)):.4f}")
print("Chuong trinh khong bao loi vi cac gia tri deu khong am.")
print("Tuy nhien gio on va diem giua ky la so lien tuc, khong phai so dem tu.")
print("Do do dung MultinomialNB la sai gia dinh phan phoi du mo hinh van chay.")


MultinomialNB: accuracy = 0.8000
GaussianNB: accuracy = 0.9000
Chuong trinh khong bao loi vi cac gia tri deu khong am.
Tuy nhien gio on va diem giua ky la so lien tuc, khong phai so dem tu.
Do do dung MultinomialNB la sai gia dinh phan phoi du mo hinh van chay.


**Nhận xét:** MultinomialNB chạy không lỗi vì các giá trị không âm, nhưng chỉ đạt accuracy=0,8000 so với 0,9000 của GaussianNB. Giờ ôn và điểm giữa kỳ là số liên tục, không phải số đếm sự kiện, nên mô hình đa thức sai về giả định dữ liệu.

## Bài tập 6. Lớp hiếm và tiên nghiệm

Giữ 180 tin thường và 20 tin rác đầu tiên; đo accuracy và recall của lớp rác.

In [6]:
"""Bài 6: thay đổi prior bằng cách chỉ giữ 20 tin rác đầu tiên."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
bo_moi = pd.concat([
    df.loc[df["la_rac"] == 0],
    df.loc[df["la_rac"] == 1].head(20),
], ignore_index=True)
tin_train, tin_test, y_train, y_test = train_test_split(
    bo_moi["noi_dung"], bo_moi["la_rac"], test_size=0.25,
    random_state=17, stratify=bo_moi["la_rac"],
)
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(tin_train)
X_test = vec.transform(tin_test)
mo_hinh = MultinomialNB(alpha=1.0).fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

print("So tin tung lop trong bo du lieu moi:")
print(bo_moi["la_rac"].value_counts().sort_index())
print(f"Prior lop rac trong tap hoc = {mo_hinh.class_count_[1] / sum(mo_hinh.class_count_):.4f}")
print(f"Ma tran: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall lop rac = {recall_score(y_test, y_pred, pos_label=1):.4f}")
print("Moc doan tat ca la thuong: accuracy = 0.9000, recall rac = 0.0000")
print("Nhan xet: accuracy de gay ao tuong tot khi 90% tin la thuong.")
print("Can xem recall va FN de biet bo loc bo sot bao nhieu tin rac.")


So tin tung lop trong bo du lieu moi:
la_rac
0    180
1     20
Name: count, dtype: int64
Prior lop rac trong tap hoc = 0.1000
Ma tran: TN=45, FP=0, FN=1, TP=4
Accuracy = 0.9800
Recall lop rac = 0.8000
Moc doan tat ca la thuong: accuracy = 0.9000, recall rac = 0.0000
Nhan xet: accuracy de gay ao tuong tot khi 90% tin la thuong.
Can xem recall va FN de biet bo loc bo sot bao nhieu tin rac.


**Nhận xét:** Tập test có 45 tin thường và 5 tin rác. Mô hình đạt accuracy=0,9800 nhưng recall rác=0,8000: vẫn lọt 1/5 tin rác. Bộ đoán mọi tin là thường cũng đạt accuracy=0,9000 nhưng recall rác=0; vì vậy phải xem recall và FN cùng với accuracy. Prior rác ở tập học chỉ còn 0,1.